# FRED Loan Return Data Download

This notebook downloads the listed FRED series, saves the raw series, aligns them on date, and writes the aligned dataset to CSV.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "data" else Path.cwd()
RAW_OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "fred_loan_return"
RAW_LONG_OUTPUT_PATH = PROJECT_ROOT / "data" / "raw" / "fred_loan_return_raw_long.csv"
ALIGNED_OUTPUT_PATH = PROJECT_ROOT / "data" / "fred_loan_return_aligned.csv"

FRED_API_URL = "https://api.stlouisfed.org/fred/series/observations"
FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv"
FRED_API_KEY = os.getenv("FRED_API_KEY")
FORCE_DOWNLOAD = False

series = {
    # Numerator of equation (12) -- domestic loans only
    "loan_interest_income": "QBPQYTIYDOFFLN",

    # Denominator -- total loans lagged one quarter
    "total_loans": "QBPBSTASTLN",

    # Charge-offs -- second term of equation (12)
    "net_chargeoffs": "QBPQYNTCGOFF",

    #charge_off_rate
    "charge_off_rate": "CORBLACBS"
}

## Download Raw FRED Series

If `FRED_API_KEY` is set in your environment, the notebook uses the official FRED API. Otherwise it falls back to FRED's public CSV endpoint.

In [2]:
def download_fred_series(alias: str, series_id: str, force_download: bool = FORCE_DOWNLOAD) -> pd.DataFrame:
    RAW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    raw_path = RAW_OUTPUT_DIR / f"{series_id}.csv"

    if raw_path.exists() and not force_download:
        raw = pd.read_csv(raw_path)
    elif FRED_API_KEY:
        params = {
            "series_id": series_id,
            "api_key": FRED_API_KEY,
            "file_type": "json",
        }
        response = requests.get(FRED_API_URL, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()
        if "observations" not in payload:
            message = payload.get("error_message", "missing observations payload")
            raise RuntimeError(f"FRED API response for {series_id} was invalid: {message}")

        raw = pd.DataFrame(
            {
                "DATE": [observation["date"] for observation in payload["observations"]],
                series_id: [observation["value"] for observation in payload["observations"]],
            }
        )
        raw.to_csv(raw_path, index=False)
    else:
        params = {"id": series_id}
        response = requests.get(FRED_CSV_URL, params=params, timeout=60)
        response.raise_for_status()
        raw_path.write_text(response.text, encoding="utf-8")
        raw = pd.read_csv(raw_path)

    raw = raw.rename(columns={"observation_date": "DATE", "date": "DATE"})
    if "DATE" not in raw.columns:
        raise ValueError(f"Downloaded data for {series_id} does not contain a DATE column")

    value_columns = [column for column in raw.columns if column != "DATE"]
    if series_id in value_columns:
        value_column = series_id
    elif len(value_columns) == 1:
        value_column = value_columns[0]
    else:
        raise ValueError(f"Downloaded data for {series_id} has ambiguous value columns: {value_columns}")

    cleaned = raw[["DATE", value_column]].rename(columns={value_column: alias})
    cleaned["DATE"] = pd.to_datetime(cleaned["DATE"], errors="coerce")
    cleaned[alias] = pd.to_numeric(cleaned[alias].replace(".", pd.NA), errors="coerce")
    cleaned = cleaned.dropna(subset=["DATE"]).sort_values("DATE").reset_index(drop=True)
    cleaned["series_id"] = series_id
    cleaned["alias"] = alias
    return cleaned


raw_frames = [download_fred_series(alias, series_id) for alias, series_id in series.items()]
raw_long = pd.concat(raw_frames, ignore_index=True)
raw_long.to_csv(RAW_LONG_OUTPUT_PATH, index=False)

print(f"Saved individual raw series to {RAW_OUTPUT_DIR}")
print(f"Saved raw long file to {RAW_LONG_OUTPUT_PATH}")
raw_long.head()

Saved individual raw series to /Users/hannahokeeffe/Documents/Imperial/DFL_New/data/raw/fred_loan_return
Saved raw long file to /Users/hannahokeeffe/Documents/Imperial/DFL_New/data/raw/fred_loan_return_raw_long.csv


,DATE,loan_interest_income,series_id,alias,total_loans,net_chargeoffs,charge_off_rate
0,1984-01-01,49811.6020,QBPQYTIYDOFFLN,loan_interest_income,NaN,NaN,NaN
1,1984-04-01,57935.4280,QBPQYTIYDOFFLN,loan_interest_income,NaN,NaN,NaN
2,1984-07-01,57521.4250,QBPQYTIYDOFFLN,loan_interest_income,NaN,NaN,NaN
3,1984-10-01,66719.1410,QBPQYTIYDOFFLN,loan_interest_income,NaN,NaN,NaN
4,1985-01-01,58933.8930,QBPQYTIYDOFFLN,loan_interest_income,NaN,NaN,NaN


## Align And Save

The aligned CSV is wide: one row per date and one column per named series.

In [3]:
aligned = None
for frame in raw_frames:
    alias = frame["alias"].iloc[0]
    wide = frame[["DATE", alias]].copy()
    aligned = wide if aligned is None else aligned.merge(wide, on="DATE", how="outer")

aligned = aligned.sort_values("DATE").reset_index(drop=True)
aligned.to_csv(ALIGNED_OUTPUT_PATH, index=False)

print(f"Saved aligned data to {ALIGNED_OUTPUT_PATH}")
aligned.tail()

Saved aligned data to /Users/hannahokeeffe/Documents/Imperial/DFL_New/data/fred_loan_return_aligned.csv


,DATE,loan_interest_income,total_loans,net_chargeoffs,charge_off_rate
164,2025-01-01,203218.1760,12787025.5450,21258.7330,0.5500
165,2025-04-01,205754.6110,13050766.6620,19390.5270,0.5800
166,2025-07-01,214587.5140,13209726.7950,20132.3110,0.5600
167,2025-10-01,214254.2460,13477526.0510,20891.3370,0.5600
168,2026-01-01,NaN,NaN,NaN,0.5900


## Coverage Check

In [4]:
coverage = aligned.agg(["count", "min", "max"]).T
coverage["missing_share"] = aligned.isna().mean()
coverage

,count,min,max,missing_share
DATE,169,1984-01-01 00:00:00,2026-01-01 00:00:00,0.0000
loan_interest_income,168.0000,49718.6650,214587.5140,0.0059
total_loans,168.0000,2034636.8160,13477526.0510,0.0059
net_chargeoffs,168.0000,1685.9360,54988.4370,0.0059
charge_off_rate,165.0000,0.1200,2.5700,0.0237
